In [22]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Descargas necesarias de NLTK para uso local
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

def ejecutar_fase_1(ruta_input="spam.csv", ruta_output="spam_limpio.csv"):
    print("Iniciando Fase 1: Limpieza y Feature Engineering...")
    
    # 1. Ingestión
    df = pd.read_csv(ruta_input, encoding='latin-1')
    
    # 2. Limpieza estructural
    columnas_basura = ['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
    df = df.drop(columns=[col for col in columnas_basura if col in df.columns])
    df = df.rename(columns={'v1': 'label', 'v2': 'texto_original'})
    df = df.drop_duplicates()
    
    # 3. Ingeniería de Características (Debe ocurrir ANTES de limpiar el texto)
    df['num_caracteres'] = df['texto_original'].apply(len).astype(np.float32)
    df['num_mayusculas'] = df['texto_original'].apply(lambda x: sum(1 for c in str(x) if c.isupper())).astype(np.float32)
    df['num_digitos'] = df['texto_original'].apply(lambda x: sum(1 for c in str(x) if c.isdigit())).astype(np.float32)
    
    # 4. Procesamiento de Lenguaje Natural (NLP)
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    def limpiar_texto(texto):
        # Remover URLs y dejar solo letras
        texto = re.sub(r'http\S+', '', str(texto))
        texto = re.sub(r'[^a-zA-Z]', ' ', texto)
        # Convertir a minúsculas y tokenizar
        tokens = texto.lower().split()
        # Filtrar stop-words y lematizar a la raíz
        tokens_limpios = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
        return ' '.join(tokens_limpios)
        
    df['texto_limpio'] = df['texto_original'].apply(limpiar_texto)
    
    # 5. Codificación de la etiqueta objetivo (0 = ham, 1 = spam)
    df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
    
    # 6. Filtrar solo las columnas útiles para el modelo
    df_final = df[['label_num', 'texto_limpio', 'num_caracteres', 'num_mayusculas', 'num_digitos']]
    
    # Exportar resultados para aislar la fase
    df_final.to_csv(ruta_output, index=False)
    print(f"Fase 1 completada. Archivo generado: {ruta_output}")
    
    return df_final

In [23]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer

def ejecutar_fase_2(ruta_input="spam_limpio.csv"):
    print("Iniciando Fase 2: Configuración de Vectorización y Partición...")
    
    # 1. Ingestión de datos limpios
    df = pd.read_csv(ruta_input)
    
    # Prevención de errores: si la limpieza de Fase 1 dejó celdas de texto vacías,
    # pandas las lee como NaN. Debemos convertirlas a strings vacíos.
    df['texto_limpio'] = df['texto_limpio'].fillna('')
    
    # 2. Separar Variables Independientes (X) y Objetivo (y)
    X = df[['texto_limpio', 'num_caracteres', 'num_mayusculas', 'num_digitos']]
    y = df['label_num']
    
    # 3. Partición Train / Test (80% / 20%)
    # stratify=y es obligatorio para asegurar que el 13% de spam se mantenga en ambas partes
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    
    # 4. Construcción del Preprocesador Matemático (El "Traductor")
    # ColumnTransformer permite aplicar reglas distintas a columnas distintas en paralelo
    preprocesador = ColumnTransformer(
        transformers=[
            # Aplica TF-IDF con un vocabulario máximo de 3000 palabras a la columna de texto
            ('texto', TfidfVectorizer(max_features=3000), 'texto_limpio'),
            # Aplica una escala de 0 a 1 a las columnas numéricas para que no dominen a las palabras
            ('numeros', MinMaxScaler(), ['num_caracteres', 'num_mayusculas', 'num_digitos'])
        ]
    )
    
    # 5. Exportación de los datos partidos (para la Fase 3)
    X_train.to_csv('X_train.csv', index=False)
    X_test.to_csv('X_test.csv', index=False)
    y_train.to_csv('y_train.csv', index=False)
    y_test.to_csv('y_test.csv', index=False)
    
    # 6. Exportación de la estructura del preprocesador
    joblib.dump(preprocesador, 'preprocesador_base.joblib')
    
    print("Fase 2 completada.")
    print(f"Dimensiones de entrenamiento: {X_train.shape[0]} filas.")
    print(f"Dimensiones de prueba: {X_test.shape[0]} filas.")

In [24]:
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

def ejecutar_fase_3(ruta_x="X_train.csv", ruta_y="y_train.csv", ruta_prep="preprocesador_base.joblib"):
    print("Iniciando Fase 3: Entrenamiento Competitivo...")
    
    # 1. Cargar datos y evitar valores nulos residuales
    X_train = pd.read_csv(ruta_x)
    X_train['texto_limpio'] = X_train['texto_limpio'].fillna('')
    
    # .values.ravel() convierte el DataFrame de una columna a un arreglo 1D que scikit-learn prefiere
    y_train = pd.read_csv(ruta_y).values.ravel() 
    
    # 2. Cargar el preprocesador de la Fase 2
    preprocesador = joblib.load(ruta_prep)
    
    # 3. Definir los algoritmos con mitigación de desbalance (class_weight='balanced')
    modelos = {
        'LogReg': LogisticRegression(class_weight='balanced', max_iter=1000),
        'NaiveBayes': MultinomialNB(), # NB maneja bien el texto por defecto
        'SVM': SVC(kernel='linear', class_weight='balanced', probability=True),
        'RandomForest': RandomForestClassifier(class_weight='balanced_subsample', random_state=42)
    }
    
    # 4. Entrenamiento y empaquetado
    for nombre, algoritmo in modelos.items():
        print(f"Entrenando candidato: {nombre}...")
        
        # El Pipeline une la transformación matemática y el algoritmo de IA en un solo paso
        pipe = Pipeline(steps=[
            ('preprocesamiento', preprocesador),
            ('clasificador', algoritmo)
        ])
        
        # Entrenar el pipeline completo
        pipe.fit(X_train, y_train)
        
        # 5. Exportar el candidato entrenado
        nombre_archivo = f"candidato_{nombre}.joblib"
        joblib.dump(pipe, nombre_archivo)
        print(f"Guardado: {nombre_archivo}")
        
    print("Fase 3 completada.")
   

In [25]:
import pandas as pd
import joblib
import json
import datetime
import os
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# Librerías para conversión a ONNX
from skl2onnx import to_onnx
from skl2onnx.common.data_types import StringTensorType, FloatTensorType

def ejecutar_fase_4(ruta_x="X_test.csv", ruta_y="y_test.csv"):
    print("Iniciando Fase 4: Evaluación y Exportación...")
    
    # 1. Cargar datos de prueba
    X_test = pd.read_csv(ruta_x)
    X_test['texto_limpio'] = X_test['texto_limpio'].fillna('')
    y_test = pd.read_csv(ruta_y).values.ravel()
    
    # Lista de los candidatos generados en Fase 3
    nombres_modelos = ['LogReg', 'NaiveBayes', 'SVM', 'RandomForest']
    
    mejor_modelo_nombre = None
    mejor_pipeline = None
    mejor_recall = -1
    metricas_ganador = {}

    # 2. Evaluación competitiva (Zero Trust)
    for nombre in nombres_modelos:
        ruta_modelo = f"candidato_{nombre}.joblib"
        if not os.path.exists(ruta_modelo):
            continue
            
        pipe = joblib.load(ruta_modelo)
        y_pred = pipe.predict(X_test)
        
        # Desempaquetar matriz de confusión
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        
        print(f"[{nombre}] FP: {fp} | Recall: {recall:.4f} | Precisión: {precision:.4f}")
        
        # 3. Lógica de selección: Falsos Positivos deben ser lo más cercanos a 0, priorizando Recall
        if fp <= 5: # Tolerancia estricta (ajustable según el negocio)
            if recall > mejor_recall:
                mejor_recall = recall
                mejor_modelo_nombre = nombre
                mejor_pipeline = pipe
                metricas_ganador = {
                    "falsos_positivos": int(fp),
                    "precision": float(precision),
                    "recall": float(recall)
                }

    if mejor_pipeline is None:
        raise ValueError("Ningún modelo cumplió con los criterios de seguridad (Demasiados Falsos Positivos).")

    print(f"\n--> Ganador seleccionado: {mejor_modelo_nombre}")

    # 4. Transcodificación a ONNX
    # Debemos decirle explícitamente a ONNX qué tipo de datos recibirá desde el JavaScript
    tipos_entrada = [
        ('texto_limpio', StringTensorType([None, 1])),
        ('num_caracteres', FloatTensorType([None, 1])),
        ('num_mayusculas', FloatTensorType([None, 1])),
        ('num_digitos', FloatTensorType([None, 1]))
    ]
    
    modelo_onnx = to_onnx(mejor_pipeline, initial_types=tipos_entrada, target_opset=12)
    nombre_archivo_onnx = "modelo_produccion.onnx"
    
    with open(nombre_archivo_onnx, "wb") as f:
        f.write(modelo_onnx.SerializeToString())
    print(f"Modelo compilado exitosamente: {nombre_archivo_onnx}")

    # 5. Actualización del version.json
    version_actual = 0
    if os.path.exists('version.json'):
        with open('version.json', 'r') as f:
            try:
                datos_version = json.load(f)
                version_actual = datos_version.get('version', 0)
            except json.JSONDecodeError:
                pass

    nueva_version = {
        "version": version_actual + 1,
        "fecha_actualizacion": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "archivo_modelo": nombre_archivo_onnx,
        "algoritmo_ganador": mejor_modelo_nombre,
        "metricas_rendimiento": metricas_ganador
    }

    with open('version.json', 'w') as f:
        json.dump(nueva_version, f, indent=4)
    print("Archivo version.json actualizado.")

In [26]:
if __name__ == "__main__":
    ejecutar_fase_1()
    ejecutar_fase_2()
    ejecutar_fase_3()
    ejecutar_fase_4()

Iniciando Fase 1: Limpieza y Feature Engineering...
Fase 1 completada. Archivo generado: spam_limpio.csv
Iniciando Fase 2: Configuración de Vectorización y Partición...
Fase 2 completada.
Dimensiones de entrenamiento: 4135 filas.
Dimensiones de prueba: 1034 filas.
Iniciando Fase 3: Entrenamiento Competitivo...
Entrenando candidato: LogReg...
Guardado: candidato_LogReg.joblib
Entrenando candidato: NaiveBayes...
Guardado: candidato_NaiveBayes.joblib
Entrenando candidato: SVM...
Guardado: candidato_SVM.joblib
Entrenando candidato: RandomForest...
Guardado: candidato_RandomForest.joblib
Fase 3 completada.
Iniciando Fase 4: Evaluación y Exportación...
[LogReg] FP: 7 | Recall: 0.9084 | Precisión: 0.9444
[NaiveBayes] FP: 2 | Recall: 0.8550 | Precisión: 0.9825
[SVM] FP: 4 | Recall: 0.9389 | Precisión: 0.9685
[RandomForest] FP: 0 | Recall: 0.9313 | Precisión: 1.0000

--> Ganador seleccionado: SVM
Modelo compilado exitosamente: modelo_produccion.onnx
Archivo version.json actualizado.
